# Exercise 4.2: Conditional Routing — Graphs That Make Decisions

**Module:** 4 — LangGraph
**Level:** Intermediate

In 4.1, you built a linear graph (research → summarize). Now we add **conditional edges** — the graph can choose different paths based on the data.

**What you'll do:**
1. Build a disruption handler that routes by severity
2. Use conditional edges to branch the graph
3. Test with different inputs to see different paths taken
4. Visualize the branching graph

**Prerequisite:** Complete 4.1 (Basic Graph) first.

## 1. Setup

Get your free Groq API key at: https://console.groq.com/keys

In [ ]:
# Install LangGraph and LangChain with Groq
!pip install langgraph langchain langchain-groq -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. The Concept: Conditional Edges

In 4.1, every edge was **fixed**: A → B → C. The graph always takes the same path.

A **conditional edge** is different. Instead of always going A → B, it runs a **routing function** that inspects the state and decides where to go next:

```
                    ┌─── high ──→ escalate ──→ END
START → analyze ────┤
                    └─── low ───→ auto_resolve → END
```

The routing function returns a **string** — the name of the next node to go to.

## 3. Define the State

Our state needs fields for each stage of the workflow.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq


class DisruptionState(TypedDict):
    # Input: the raw disruption report
    disruption: str
    # Filled by analyze_node: the severity level
    severity: str           # "low", "medium", or "high"
    # Filled by analyze_node: detailed analysis
    analysis: str
    # Filled by the handler node (whichever one runs): the resolution
    resolution: str
    # Filled by notify_node: passenger notification
    notification: str


# Create the LLM
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("State defined with 5 fields.")
print("The 'severity' field is what the routing function will check.")

## 4. Define the Nodes

We need four nodes:
1. **analyze** — classify the disruption severity
2. **auto_resolve** — handle minor issues automatically
3. **escalate** — handle severe issues with urgency
4. **notify** — generate the passenger notification

In [ ]:
def analyze_node(state: DisruptionState) -> dict:
    """Analyze the disruption and determine severity."""
    disruption = state["disruption"]

    # Ask the LLM to classify severity AND provide analysis
    response = model.invoke(
        f"""Analyze this flight disruption. You MUST respond in this EXACT format:

SEVERITY: [low/medium/high]
ANALYSIS: [your detailed analysis]

Rules for severity:
- low: delays under 1 hour, minor issues
- medium: delays 1-3 hours, some passengers affected
- high: cancellations, delays over 3 hours, many connecting passengers

Disruption:
{disruption}"""
    )

    content = response.content

    # Parse the severity from the response
    # We look for "SEVERITY: low/medium/high" in the LLM output
    severity = "medium"  # default if parsing fails
    for line in content.split("\n"):
        if line.strip().upper().startswith("SEVERITY:"):
            parsed = line.split(":", 1)[1].strip().lower()
            if parsed in ["low", "medium", "high"]:
                severity = parsed
            break

    return {
        "severity": severity,
        "analysis": content
    }


print("analyze_node: classifies severity as low/medium/high")

In [ ]:
def auto_resolve_node(state: DisruptionState) -> dict:
    """Handle minor disruptions automatically."""
    analysis = state["analysis"]

    response = model.invoke(
        f"""This is a LOW/MEDIUM severity disruption. Provide a quick resolution.
Be concise — 2-3 bullet points max. Focus on standard procedures.

Analysis:
{analysis}"""
    )

    return {"resolution": f"[AUTO-RESOLVED]\n{response.content}"}


def escalate_node(state: DisruptionState) -> dict:
    """Handle severe disruptions with urgency."""
    analysis = state["analysis"]

    response = model.invoke(
        f"""URGENT: This is a HIGH severity disruption. Provide emergency resolution.
Include:
1. Immediate actions (rebooking, hotel arrangements)
2. Customer care team activation
3. Communication plan

Analysis:
{analysis}"""
    )

    return {"resolution": f"[ESCALATED — URGENT]\n{response.content}"}


print("auto_resolve_node: handles minor issues with standard procedures")
print("escalate_node: handles severe issues with emergency response")

In [ ]:
def notify_node(state: DisruptionState) -> dict:
    """Generate passenger notification based on the resolution."""
    resolution = state["resolution"]
    severity = state["severity"]

    # Adjust the tone based on severity
    tone = "empathetic and urgent" if severity == "high" else "friendly and reassuring"

    response = model.invoke(
        f"""Write a {tone} passenger notification based on this resolution.
Max 3 sentences. Include next steps for the passenger.

Resolution:
{resolution}"""
    )

    return {"notification": response.content}


print("notify_node: creates passenger notification (always runs at the end)")

## 5. The Routing Function

This is the key piece. The routing function **inspects the state** and returns the **name of the next node**. LangGraph calls this function at the conditional edge to decide the path.

In [ ]:
def route_by_severity(state: DisruptionState) -> str:
    """Routing function: decides which handler to use based on severity.

    Returns the NAME of the next node to execute.
    LangGraph uses this return value to follow the correct edge.
    """
    severity = state["severity"]

    if severity == "high":
        print(f"  🔴 Severity: {severity} → routing to ESCALATE")
        return "escalate"       # Go to escalate_node
    else:
        print(f"  🟢 Severity: {severity} → routing to AUTO-RESOLVE")
        return "auto_resolve"   # Go to auto_resolve_node


print("Routing function defined.")
print("It reads state['severity'] and returns the next node name.")

## 6. Build the Graph with Conditional Edges

Now we connect everything. The key new method is `add_conditional_edges()` — instead of a fixed connection, it calls the routing function.

In [ ]:
# Create the graph
builder = StateGraph(DisruptionState)

# Add all four nodes
builder.add_node("analyze", analyze_node)
builder.add_node("auto_resolve", auto_resolve_node)
builder.add_node("escalate", escalate_node)
builder.add_node("notify", notify_node)

# Add edges
builder.add_edge(START, "analyze")  # Always start with analysis

# CONDITIONAL EDGE: after analyze, call route_by_severity to decide next node
builder.add_conditional_edges(
    "analyze",              # From this node...
    route_by_severity,      # ...call this function to decide where to go
    {                        # Map return values to node names:
        "auto_resolve": "auto_resolve",  # If function returns "auto_resolve" → go to auto_resolve node
        "escalate": "escalate"            # If function returns "escalate" → go to escalate node
    }
)

# Both handlers lead to the notification step
builder.add_edge("auto_resolve", "notify")  # After auto-resolve → notify
builder.add_edge("escalate", "notify")      # After escalate → notify
builder.add_edge("notify", END)              # After notify → done

# Compile
graph = builder.compile()
print("Graph compiled with conditional routing!")

In [ ]:
# Visualize the graph — you should see the branching after "analyze"
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph structure:")
    print("START → analyze")
    print("  analyze ──(high)──→ escalate ──→ notify → END")
    print("  analyze ──(low/med)→ auto_resolve → notify → END")

## 7. Test Case 1: Minor Delay (Low Severity)

A short delay should be auto-resolved.

In [ ]:
# Test 1: Minor delay — should take the auto_resolve path
minor_disruption = {
    "disruption": """Flight TK2001 from Istanbul (IST) to Frankfurt (FRA):
- Scheduled departure: 10:00
- Current status: Delayed 30 minutes (late incoming aircraft)
- 150 passengers booked
- No connecting flight issues expected"""
}

print("=== Test 1: Minor Delay ===")
print("Input:", minor_disruption["disruption"][:80], "...")
print()

result = graph.invoke(minor_disruption)

print(f"\nSeverity: {result['severity']}")
print(f"\n--- Resolution ---")
print(result["resolution"])
print(f"\n--- Passenger Notification ---")
print(result["notification"])

## 8. Test Case 2: Flight Cancellation (High Severity)

A cancellation should be escalated.

In [ ]:
# Test 2: Cancellation — should take the escalate path
major_disruption = {
    "disruption": """Flight TK7890 from Istanbul (IST) to New York (JFK):
- Scheduled departure: 23:00
- Current status: CANCELLED (engine inspection required)
- 320 passengers booked (including 85 premium class)
- 120 passengers have connecting flights at JFK
- Next available IST-JFK flight: 2 days later (full)
- Alternative routing possible via London (LHR)"""
}

print("=== Test 2: Flight Cancellation ===")
print("Input:", major_disruption["disruption"][:80], "...")
print()

result = graph.invoke(major_disruption)

print(f"\nSeverity: {result['severity']}")
print(f"\n--- Resolution ---")
print(result["resolution"])
print(f"\n--- Passenger Notification ---")
print(result["notification"])

## 9. Streaming — Watch the Path in Real Time

Let's use `.stream()` to see exactly which nodes execute and in what order.

In [ ]:
# Stream execution to see which path is taken
medium_disruption = {
    "disruption": """Flight TK4567 from Istanbul (IST) to Rome (FCO):
- Scheduled departure: 15:30
- Current status: Delayed 2 hours (crew scheduling issue)
- 175 passengers booked
- 20 passengers have tight connections at FCO"""
}

print("=== Streaming Execution ===")
print("Watch which nodes execute:\n")

for step in graph.stream(medium_disruption):
    node_name = list(step.keys())[0]
    print(f">>> Node '{node_name}' completed")
    # Show which fields were updated
    for key, value in step[node_name].items():
        preview = value[:100] + "..." if len(str(value)) > 100 else value
        print(f"    Updated: {key} = {preview}")
    print()

---

## YOUR TURN: Add a New Condition

Extend the graph with a **medium** severity handler. The routing should now have three paths:

```
analyze ──(high)──→ escalate → notify → END
analyze ──(medium)→ review   → notify → END
analyze ──(low)───→ auto_resolve → notify → END
```

The `review` node should suggest having a supervisor review the situation before deciding.

Hints:
1. Create a `review_node` function
2. Update `route_by_severity` to return `"review"` for medium
3. Add the new node and edges to the graph

In [ ]:
# YOUR CODE HERE

# 1. Define a review_node function
# def review_node(state) -> dict:
#     ...
#     return {"resolution": f"[UNDER REVIEW]\n{response.content}"}

# 2. Update routing function to handle 3 levels
# def route_three_levels(state) -> str:
#     if severity == "high": return "escalate"
#     elif severity == "medium": return "review"
#     else: return "auto_resolve"

# 3. Build the graph with 3 branches
# builder = StateGraph(DisruptionState)
# ...

# 4. Test with all three severity levels



## Key Takeaways

- **Conditional edges** let graphs make decisions based on state data
- **Routing function** inspects state and returns the name of the next node
- **`add_conditional_edges(source, func, mapping)`** — the key method
- Multiple paths can **converge** to the same node (both handlers → notify)
- Use **`.stream()`** to debug which path the graph takes

**Next:** In Exercise 5.1, you'll build a **complete agent** that combines everything — state, nodes, conditional routing, and error handling.